In [1]:
# ============================================================
# Step 6 setup
#
# Upload:
#   1. config.py
#
# Read from Google Drive:
#   2. Step 5 hidden-state zip
#
# This cell prepares:
#   /content/pilot_4/config.py
#   cfg.STEP6_INPUT_DIR
#   cfg.STEP6_OUTPUT_DIR
# ============================================================

import os
import sys
import shutil
import zipfile
import importlib
from pathlib import Path

from google.colab import files
from google.colab import drive

drive.mount("/content/drive")

# ------------------------------------------------------------
# 1. Create project structure
# ------------------------------------------------------------

PILOT_ROOT = "/content/pilot_4"
SCRIPTS_DIR = os.path.join(PILOT_ROOT, "scripts")
DATA_DIR = os.path.join(PILOT_ROOT, "data")

os.makedirs(PILOT_ROOT, exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

with open(os.path.join(PILOT_ROOT, "__init__.py"), "w", encoding="utf-8") as f:
    f.write("")

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# ------------------------------------------------------------
# 2. Upload config.py
# ------------------------------------------------------------

print("Upload config.py")
uploaded_config = files.upload()

config_upload_name = None

for name in uploaded_config.keys():
    if name.startswith("config") and name.endswith(".py"):
        config_upload_name = name
        break

if config_upload_name is None:
    raise FileNotFoundError(
        f"No config*.py file was uploaded. Uploaded files: {list(uploaded_config.keys())}"
    )

config_target_path = os.path.join(PILOT_ROOT, "config.py")

shutil.copy(
    os.path.join("/content", config_upload_name),
    config_target_path,
)

# ------------------------------------------------------------
# 3. Load config
# ------------------------------------------------------------

import pilot_4.config as cfg
importlib.reload(cfg)

print("\nLoaded config:")
print("PILOT_ROOT:", cfg.PILOT_ROOT)
print("DATA_DIR:", cfg.DATA_DIR)
print("STEP6_INPUT_DIR:", cfg.STEP6_INPUT_DIR)
print("STEP6_OUTPUT_DIR:", cfg.STEP6_OUTPUT_DIR)
print("STEP6_EXPERIMENT_NAME:", cfg.STEP6_EXPERIMENT_NAME)

# ------------------------------------------------------------
# 4. Prepare Step 6 input/output directories
# ------------------------------------------------------------

os.makedirs(cfg.STEP6_INPUT_DIR, exist_ok=True)

# Clear old Step 6 input files.
for old_path in Path(cfg.STEP6_INPUT_DIR).glob("*"):
    if old_path.is_file():
        old_path.unlink()
    elif old_path.is_dir():
        shutil.rmtree(old_path)

# Clear old Step 6 outputs.
if os.path.exists(cfg.STEP6_OUTPUT_DIR):
    shutil.rmtree(cfg.STEP6_OUTPUT_DIR)

os.makedirs(cfg.STEP6_OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 5. Use Step 5 zip from Google Drive
# ------------------------------------------------------------

step5_zip_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expA_settingA_6fps/settingA/data_expA_6fps_settingA/p4_settingA_step5_hidden_states_Qwen2_5_3B.zip"
)

print("\nUsing Step 5 zip from Google Drive:")
print(step5_zip_path)

if not step5_zip_path.exists():
    raise FileNotFoundError(f"Step 5 zip not found: {step5_zip_path}")

# ------------------------------------------------------------
# 6. Unzip Step 5 .pt files into STEP6_INPUT_DIR
# ------------------------------------------------------------

with zipfile.ZipFile(str(step5_zip_path), "r") as zf:
    zf.extractall(cfg.STEP6_INPUT_DIR)

pt_files = sorted(Path(cfg.STEP6_INPUT_DIR).rglob("*.pt"))

if not pt_files:
    raise FileNotFoundError(
        f"No .pt files found after extracting Step 5 zip into {cfg.STEP6_INPUT_DIR}"
    )

print("\nSetup complete.")
print("Config copied to:", config_target_path)
print("Step 5 zip:", step5_zip_path)
print("Number of .pt files found:", len(pt_files))
print("STEP6_INPUT_DIR:", cfg.STEP6_INPUT_DIR)
print("STEP6_OUTPUT_DIR:", cfg.STEP6_OUTPUT_DIR)

for p in pt_files[:10]:
    print("-", p)

Mounted at /content/drive
Upload config.py


Saving config_p4_expA_settingB3.py to config_p4_expA_settingB3.py

Loaded config:
PILOT_ROOT: /content/pilot_4
DATA_DIR: /content/pilot_4/data
STEP6_INPUT_DIR: /content/pilot_4/data/step5_hidden_states_input
STEP6_OUTPUT_DIR: /content/pilot_4/data/step6_settingB3_scene_split_train_direct_test_direct_outputs
STEP6_EXPERIMENT_NAME: settingB3_scene_split_train_direct_test_direct

Using Step 5 zip from Google Drive:
/content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expA_settingA_6fps/settingA/data_expA_6fps_settingA/p4_settingA_step5_hidden_states_Qwen2_5_3B.zip

Setup complete.
Config copied to: /content/pilot_4/config.py
Step 5 zip: /content/drive/MyDrive/Colab Notebooks/linear_probe_pilot4/expA_settingA_6fps/settingA/data_expA_6fps_settingA/p4_settingA_step5_hidden_states_Qwen2_5_3B.zip
Number of .pt files found: 6
STEP6_INPUT_DIR: /content/pilot_4/data/step5_hidden_states_input
STEP6_OUTPUT_DIR: /content/pilot_4/data/step6_settingB3_scene_split_train_direct_test_direct_outputs

In [2]:
# ============================================================
# Imports and Step 6 configuration
# ============================================================

import os
import sys
import json
import random
import importlib
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import pilot_4.config as cfg
importlib.reload(cfg)

# ------------------------------------------------------------
# Basic config
# ------------------------------------------------------------

EXPERIMENT_NAME = cfg.STEP6_EXPERIMENT_NAME
RANDOM_SEED = getattr(cfg, "RANDOM_SEED", 42)

MODEL_NAME = getattr(cfg, "STEP5_MODEL_NAME", "Qwen/Qwen2.5-3B")
MODEL_TAG = getattr(cfg, "STEP5_MODEL_TAG", "Qwen2_5_3B")

STEP6_INPUT_DIR = cfg.STEP6_INPUT_DIR
STEP6_OUTPUT_DIR = cfg.STEP6_OUTPUT_DIR

FEATURE_KEY = cfg.STEP6_FEATURE_KEY
LABEL_FIELD = cfg.STEP6_LABEL_FIELD

TRAIN_SCENES = list(cfg.STEP6_TRAIN_SCENES)
TEST_SCENES = list(cfg.STEP6_TEST_SCENES)
REQUIRE_EXPLICIT_SCENE_SPLIT = getattr(cfg, "STEP6_REQUIRE_EXPLICIT_SCENE_SPLIT", True)

LOGREG_MAX_ITER = getattr(cfg, "STEP6_LOGREG_MAX_ITER", 5000)
LOGREG_C = getattr(cfg, "STEP6_LOGREG_C", 1.0)

EXPLICIT_DIRECT = cfg.EXPLICIT_DIRECT
EXPLICIT_INVERSE = cfg.EXPLICIT_INVERSE

PRINT_DATASET_SUMMARY = getattr(cfg, "STEP6_PRINT_DATASET_SUMMARY", True)
PRINT_LAYER_PROGRESS = getattr(cfg, "STEP6_PRINT_LAYER_PROGRESS", True)
PRINT_TOP_LAYERS = getattr(cfg, "STEP6_PRINT_TOP_LAYERS", True)
NUM_TOP_LAYERS_TO_PRINT = getattr(cfg, "STEP6_NUM_TOP_LAYERS_TO_PRINT", 10)

# ------------------------------------------------------------
# Setting B3 same-direction filtering config
# ------------------------------------------------------------

DIRECTION_FILTER_PROTOCOL = getattr(
    cfg,
    "STEP6_DIRECTION_FILTER_PROTOCOL",
    None,
)

TRAIN_DIRECTION_SELECTION_MODE = getattr(
    cfg,
    "STEP6_TRAIN_DIRECTION_SELECTION_MODE",
    None,
)

TEST_DIRECTION_SELECTION_MODE = getattr(
    cfg,
    "STEP6_TEST_DIRECTION_SELECTION_MODE",
    None,
)

FILTER_TRAIN_BY_DIRECTION = getattr(
    cfg,
    "STEP6_FILTER_TRAIN_BY_DIRECTION",
    None,
)

FILTER_TEST_BY_DIRECTION = getattr(
    cfg,
    "STEP6_FILTER_TEST_BY_DIRECTION",
    None,
)

# Setting B3 notebook must not silently fall back to unfiltered behavior.
if DIRECTION_FILTER_PROTOCOL != "same_direction":
    raise ValueError(
        "This notebook is Setting B3 same-direction only. "
        "Config must set STEP6_DIRECTION_FILTER_PROTOCOL = 'same_direction'."
    )

if TRAIN_DIRECTION_SELECTION_MODE not in {"direct", "inverse"}:
    raise ValueError(
        "Setting B3 requires STEP6_TRAIN_DIRECTION_SELECTION_MODE "
        "to be either 'direct' or 'inverse'."
    )

if TEST_DIRECTION_SELECTION_MODE not in {"direct", "inverse"}:
    raise ValueError(
        "Setting B3 requires STEP6_TEST_DIRECTION_SELECTION_MODE "
        "to be either 'direct' or 'inverse'."
    )

if TRAIN_DIRECTION_SELECTION_MODE != TEST_DIRECTION_SELECTION_MODE:
    raise ValueError(
        "Setting B3 same-direction evaluation requires train and test directions to be the same."
    )

if FILTER_TRAIN_BY_DIRECTION is not True:
    raise ValueError(
        "Setting B3 requires STEP6_FILTER_TRAIN_BY_DIRECTION = True."
    )

if FILTER_TEST_BY_DIRECTION is not True:
    raise ValueError(
         "Setting B3 requires STEP6_FILTER_TEST_BY_DIRECTION = True."
    )

print("\nSetting B3 same-direction filtering config:")
print("DIRECTION_FILTER_PROTOCOL:", DIRECTION_FILTER_PROTOCOL)
print("TRAIN_DIRECTION_SELECTION_MODE:", TRAIN_DIRECTION_SELECTION_MODE)
print("TEST_DIRECTION_SELECTION_MODE:", TEST_DIRECTION_SELECTION_MODE)
print("FILTER_TRAIN_BY_DIRECTION:", FILTER_TRAIN_BY_DIRECTION)
print("FILTER_TEST_BY_DIRECTION:", FILTER_TEST_BY_DIRECTION)


Setting B3 same-direction filtering config:
DIRECTION_FILTER_PROTOCOL: same_direction
TRAIN_DIRECTION_SELECTION_MODE: direct
TEST_DIRECTION_SELECTION_MODE: direct
FILTER_TRAIN_BY_DIRECTION: True
FILTER_TEST_BY_DIRECTION: True


In [3]:
# ============================================================
# Helper functions for loading Step 5 records
# ============================================================

def discover_step5_pt_files(input_dir: str) -> List[str]:
    pt_paths = sorted(Path(input_dir).rglob("*.pt"))

    if not pt_paths:
        raise FileNotFoundError(f"No .pt files found in STEP6_INPUT_DIR: {input_dir}")

    return [str(p) for p in pt_paths]


def normalize_record(raw_record: Dict[str, Any], source_pt_file: str) -> Dict[str, Any]:
    """
    Normalize one Step 5 record into a predictable structure.

    Expected fields:
      - FEATURE_KEY, e.g. layer_diff_features
      - LABEL_FIELD, e.g. relation
      - metadata fields such as scene, evidence_type, pair_group_id
    """

    rec = dict(raw_record)
    rec["_source_pt_file"] = source_pt_file

    # Some versions save metadata as rec["metadata"], while others flatten it.
    metadata = rec.get("metadata", {})

    if metadata is None:
        metadata = {}

    if not isinstance(metadata, dict):
        metadata = {}

    # Flatten useful metadata into rec if absent.
    for k, v in metadata.items():
        rec.setdefault(k, v)

    # If Step 5 saved nested evidence/probe_pair, expose useful fields.
    evidence = rec.get("evidence", {})
    if isinstance(evidence, dict):
        rec.setdefault("evidence_type", evidence.get("evidence_type"))
        rec.setdefault("probe_direction_relative_to_text", evidence.get("probe_direction_relative_to_text"))
        rec.setdefault("text_relation_label", evidence.get("text_relation_label"))
        rec.setdefault("evidence_sentence", evidence.get("evidence_sentence"))

    probe_pair = rec.get("probe_pair", {})
    if isinstance(probe_pair, dict):
        rec.setdefault("probe_subject_uid", probe_pair.get("probe_subject_uid"))
        rec.setdefault("probe_object_uid", probe_pair.get("probe_object_uid"))
        rec.setdefault("probe_relation_label", probe_pair.get("probe_relation_label"))
        rec.setdefault("is_inverse_of_text_relation", probe_pair.get("is_inverse_of_text_relation"))

    # Standardize relation label.
    if LABEL_FIELD not in rec:
        if "relation" in rec:
            rec[LABEL_FIELD] = rec["relation"]
        elif "probe_relation_label" in rec:
            rec[LABEL_FIELD] = rec["probe_relation_label"]
        elif isinstance(probe_pair, dict) and "probe_relation_label" in probe_pair:
            rec[LABEL_FIELD] = probe_pair["probe_relation_label"]
        else:
            raise KeyError(f"Could not find label field '{LABEL_FIELD}' in record.")

    # Standardize scene.
    if "scene" not in rec:
        if "source_step3_scene" in rec:
            rec["scene"] = rec["source_step3_scene"]
        else:
            raise KeyError("Could not find scene field in record.")

    return rec


def extract_records_from_payload(payload: Any, source_pt_file: str) -> List[Dict[str, Any]]:
    """
    Supports several possible Step 5 payload formats:
      - list[record]
      - dict with key "records"
      - dict with key "data"
    """

    if isinstance(payload, list):
        raw_records = payload
    elif isinstance(payload, dict):
        if "records" in payload:
            raw_records = payload["records"]
        elif "data" in payload:
            raw_records = payload["data"]
        elif "examples" in payload:
            raw_records = payload["examples"]
        else:
            raise KeyError(
                f"Unsupported .pt payload keys in {source_pt_file}: {list(payload.keys())}"
            )
    else:
        raise TypeError(f"Unsupported .pt payload type in {source_pt_file}: {type(payload)}")

    records = [
        normalize_record(r, source_pt_file=source_pt_file)
        for r in raw_records
    ]

    return records


def to_numpy_feature(x: Any) -> np.ndarray:
    if isinstance(x, torch.Tensor):
        # NumPy cannot directly convert torch.bfloat16 tensors.
        # Move to CPU and convert to float32 before calling numpy().
        return x.detach().cpu().float().numpy()

    arr = np.asarray(x)

    if arr.dtype == np.dtype("O"):
        arr = arr.astype(np.float32)

    return arr.astype(np.float32, copy=False)

In [4]:
# ============================================================
# Load Step 5 records and build feature tensor
# ============================================================

pt_files = discover_step5_pt_files(STEP6_INPUT_DIR)

all_records = []

for pt_path in pt_files:
    payload = torch.load(pt_path, map_location="cpu")
    records = extract_records_from_payload(payload, source_pt_file=pt_path)
    all_records.extend(records)

if not all_records:
    raise ValueError("No Step 5 records loaded.")

print("Number of Step 5 records loaded:", len(all_records))
print("Number of source .pt files:", len(pt_files))

# ------------------------------------------------------------
# Validate feature and label fields
# ------------------------------------------------------------

missing_feature = [i for i, r in enumerate(all_records) if FEATURE_KEY not in r]
missing_label = [i for i, r in enumerate(all_records) if LABEL_FIELD not in r]

if missing_feature:
    raise KeyError(f"{len(missing_feature)} records are missing feature key: {FEATURE_KEY}")

if missing_label:
    raise KeyError(f"{len(missing_label)} records are missing label field: {LABEL_FIELD}")

# ------------------------------------------------------------
# Stack features
# Expected shape per record: [num_layers, hidden_dim]
# ------------------------------------------------------------

features = []

for r in all_records:
    f = to_numpy_feature(r[FEATURE_KEY])

    if f.ndim != 2:
        raise ValueError(
            f"Expected feature shape [num_layers, dim], got {f.shape} "
            f"for sample_id={r.get('sample_id')}"
        )

    features.append(f)

X_all = np.stack(features, axis=0)  # [num_samples, num_layers, dim]
y_all = np.array([r[LABEL_FIELD] for r in all_records])
metadata = all_records

num_samples, num_layers, feature_dim = X_all.shape

print("X_all shape:", X_all.shape)
print("num_samples:", num_samples)
print("num_layers:", num_layers)
print("feature_dim:", feature_dim)
print("Label counts:")
print(dict(Counter(y_all)))

print("\nScene counts:")
print(dict(Counter(r["scene"] for r in metadata)))

print("\nEvidence type counts:")
print(dict(Counter(r.get("evidence_type", "unknown") for r in metadata)))

Number of Step 5 records loaded: 3782
Number of source .pt files: 6
X_all shape: (3782, 37, 2048)
num_samples: 3782
num_layers: 37
feature_dim: 2048
Label counts:
{np.str_('above'): 330, np.str_('right_of'): 951, np.str_('contains'): 48, np.str_('on'): 435, np.str_('supports'): 435, np.str_('below'): 330, np.str_('in'): 48, np.str_('left_of'): 951, np.str_('near'): 254}

Scene counts:
{'FloorPlan1': 708, 'FloorPlan2': 582, 'FloorPlan3': 564, 'FloorPlan4': 580, 'FloorPlan5': 708, 'FloorPlan6': 640}

Evidence type counts:
{'explicit_inverse_or_same_sentence_pair': 1872, 'explicit_direct': 1910}


In [5]:
# ============================================================
# Scene split and initial common-label filtering
# ============================================================

all_scenes = sorted(set(r["scene"] for r in metadata))

if REQUIRE_EXPLICIT_SCENE_SPLIT:
    missing_train = sorted(set(TRAIN_SCENES) - set(all_scenes))
    missing_test = sorted(set(TEST_SCENES) - set(all_scenes))

    if missing_train or missing_test:
        raise ValueError(
            f"Scene split references missing scenes. "
            f"missing_train={missing_train}, missing_test={missing_test}, "
            f"available_scenes={all_scenes}"
        )

train_idx = np.array(
    [i for i, r in enumerate(metadata) if r["scene"] in TRAIN_SCENES],
    dtype=np.int64,
)

test_idx = np.array(
    [i for i, r in enumerate(metadata) if r["scene"] in TEST_SCENES],
    dtype=np.int64,
)

if len(train_idx) == 0:
    raise ValueError("No training examples selected.")

if len(test_idx) == 0:
    raise ValueError("No test examples selected.")

# Initial common-label filter before direction filtering.
train_labels_before = set(y_all[train_idx])
test_labels_before = set(y_all[test_idx])
common_labels_initial = sorted(train_labels_before & test_labels_before)

removed_train_only_labels_initial = sorted(train_labels_before - test_labels_before)
removed_test_only_labels_initial = sorted(test_labels_before - train_labels_before)

train_idx = np.array(
    [idx for idx in train_idx if y_all[idx] in common_labels_initial],
    dtype=np.int64,
)

test_idx = np.array(
    [idx for idx in test_idx if y_all[idx] in common_labels_initial],
    dtype=np.int64,
)

scene_split_info = {
    "train_scenes": TRAIN_SCENES,
    "test_scenes": TEST_SCENES,
    "num_train_before_common_label_filter": int(sum(r["scene"] in TRAIN_SCENES for r in metadata)),
    "num_test_before_common_label_filter": int(sum(r["scene"] in TEST_SCENES for r in metadata)),
    "train_labels_before": sorted(train_labels_before),
    "test_labels_before": sorted(test_labels_before),
    "common_labels_initial": common_labels_initial,
    "removed_train_only_labels_initial": removed_train_only_labels_initial,
    "removed_test_only_labels_initial": removed_test_only_labels_initial,
    "num_train_after_initial_common_label_filter": int(len(train_idx)),
    "num_test_after_initial_common_label_filter": int(len(test_idx)),
}

print("Initial scene split complete.")
print(json.dumps(scene_split_info, indent=2, ensure_ascii=False))

print("\nTrain label counts after initial common-label filter:")
print(dict(Counter(y_all[train_idx])))

print("\nTest label counts after initial common-label filter:")
print(dict(Counter(y_all[test_idx])))

print("\nTrain evidence type counts:")
print(dict(Counter(metadata[idx].get("evidence_type", "unknown") for idx in train_idx)))

print("\nTest evidence type counts:")
print(dict(Counter(metadata[idx].get("evidence_type", "unknown") for idx in test_idx)))

Initial scene split complete.
{
  "train_scenes": [
    "FloorPlan1",
    "FloorPlan2",
    "FloorPlan3",
    "FloorPlan4"
  ],
  "test_scenes": [
    "FloorPlan5",
    "FloorPlan6"
  ],
  "num_train_before_common_label_filter": 2434,
  "num_test_before_common_label_filter": 1348,
  "train_labels_before": [
    "above",
    "below",
    "contains",
    "in",
    "left_of",
    "near",
    "on",
    "right_of",
    "supports"
  ],
  "test_labels_before": [
    "above",
    "below",
    "contains",
    "in",
    "left_of",
    "near",
    "on",
    "right_of",
    "supports"
  ],
  "common_labels_initial": [
    "above",
    "below",
    "contains",
    "in",
    "left_of",
    "near",
    "on",
    "right_of",
    "supports"
  ],
  "removed_train_only_labels_initial": [],
  "removed_test_only_labels_initial": [],
  "num_train_after_initial_common_label_filter": 2434,
  "num_test_after_initial_common_label_filter": 1348
}

Train label counts after initial common-label filter:
{np.str_('a

In [6]:
# ============================================================
# Setting B3 same-direction filtering only
#
# Protocol:
#   train direct  -> test direct
#   train inverse -> test inverse
#
# Scene split:
#   train: FloorPlan1-4
#   test:  FloorPlan5-6
# ============================================================

def direction_mode_to_evidence_type(mode: str) -> str:
    """
    Map Setting B3 direction mode to Step 4 evidence_type.
    """
    if mode == "direct":
        return EXPLICIT_DIRECT

    if mode == "inverse":
        return EXPLICIT_INVERSE

    raise ValueError(f"Unknown Setting B3 direction mode: {mode}")


def filter_indices_by_settingB3_direction(
    indices: np.ndarray,
    metadata: List[Dict[str, Any]],
    mode: str,
    split_name: str,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    """
    Keep only examples whose evidence_type matches the requested direction.

    This is the only direction filter used in Setting B3 same-direction evaluation.
    """
    target_evidence_type = direction_mode_to_evidence_type(mode)

    before_counts = Counter(
        metadata[int(idx)].get("evidence_type", "unknown")
        for idx in indices
    )

    kept_indices = np.array(
        [
            int(idx)
            for idx in indices
            if metadata[int(idx)].get("evidence_type") == target_evidence_type
        ],
        dtype=np.int64,
    )

    after_counts = Counter(
        metadata[int(idx)].get("evidence_type", "unknown")
        for idx in kept_indices
    )

    info = {
        "setting": "B3_same_direction_scene_split",
        "split": split_name,
        "mode": mode,
        "target_evidence_type": target_evidence_type,
        "num_before": int(len(indices)),
        "num_after": int(len(kept_indices)),
        "evidence_type_counts_before": dict(before_counts),
        "evidence_type_counts_after": dict(after_counts),
    }

    return kept_indices, info


train_idx_before_direction_filter = train_idx.copy()
test_idx_before_direction_filter = test_idx.copy()

train_idx, train_direction_filter_info = filter_indices_by_settingB3_direction(
    indices=train_idx,
    metadata=metadata,
    mode=TRAIN_DIRECTION_SELECTION_MODE,
    split_name="train",
)

test_idx, test_direction_filter_info = filter_indices_by_settingB3_direction(
    indices=test_idx,
    metadata=metadata,
    mode=TEST_DIRECTION_SELECTION_MODE,
    split_name="test",
)

print("Setting B3 same-direction filtering complete.")

print("\nTrain direction filter info:")
print(json.dumps(train_direction_filter_info, indent=2, ensure_ascii=False))

print("\nTest direction filter info:")
print(json.dumps(test_direction_filter_info, indent=2, ensure_ascii=False))

if len(train_idx) == 0:
    raise ValueError(
        f"No training examples left after Setting B3 same-direction filtering: "
        f"train mode = {TRAIN_DIRECTION_SELECTION_MODE}."
    )

if len(test_idx) == 0:
    raise ValueError(
        f"No test examples left after Setting B3 same-direction filtering: "
        f"test mode = {TEST_DIRECTION_SELECTION_MODE}."
    )

Setting B3 same-direction filtering complete.

Train direction filter info:
{
  "setting": "B3_same_direction_scene_split",
  "split": "train",
  "mode": "direct",
  "target_evidence_type": "explicit_direct",
  "num_before": 2434,
  "num_after": 1232,
  "evidence_type_counts_before": {
    "explicit_inverse_or_same_sentence_pair": 1202,
    "explicit_direct": 1232
  },
  "evidence_type_counts_after": {
    "explicit_direct": 1232
  }
}

Test direction filter info:
{
  "setting": "B3_same_direction_scene_split",
  "split": "test",
  "mode": "direct",
  "target_evidence_type": "explicit_direct",
  "num_before": 1348,
  "num_after": 678,
  "evidence_type_counts_before": {
    "explicit_inverse_or_same_sentence_pair": 670,
    "explicit_direct": 678
  },
  "evidence_type_counts_after": {
    "explicit_direct": 678
  }
}


In [7]:
# ============================================================
# Final label consistency after direction filtering
#
# Direction filtering may remove labels from train or test.
# Recompute common labels after both train/test direction filters.
# ============================================================

train_labels_after_direction = set(y_all[train_idx])
test_labels_after_direction = set(y_all[test_idx])

common_labels_final = sorted(train_labels_after_direction & test_labels_after_direction)

removed_train_only_labels_final = sorted(train_labels_after_direction - test_labels_after_direction)
removed_test_only_labels_final = sorted(test_labels_after_direction - train_labels_after_direction)

train_idx = np.array(
    [int(idx) for idx in train_idx if y_all[int(idx)] in common_labels_final],
    dtype=np.int64,
)

test_idx = np.array(
    [int(idx) for idx in test_idx if y_all[int(idx)] in common_labels_final],
    dtype=np.int64,
)

if len(train_idx) == 0:
    raise ValueError("No training examples left after final common-label filtering.")

if len(test_idx) == 0:
    raise ValueError("No test examples left after final common-label filtering.")

label_order = common_labels_final

scene_split_info.update({
    "setting": "B3_same_direction_scene_split",

    "direction_filter_protocol": DIRECTION_FILTER_PROTOCOL,

    "train_direction_selection_mode": TRAIN_DIRECTION_SELECTION_MODE,
    "test_direction_selection_mode": TEST_DIRECTION_SELECTION_MODE,

    "filter_train_by_direction": FILTER_TRAIN_BY_DIRECTION,
    "filter_test_by_direction": FILTER_TEST_BY_DIRECTION,

    "train_direction_filter_info": train_direction_filter_info,
    "test_direction_filter_info": test_direction_filter_info,

    "train_labels_after_direction_filter": sorted(train_labels_after_direction),
    "test_labels_after_direction_filter": sorted(test_labels_after_direction),

    "common_labels_final": common_labels_final,
    "removed_train_only_labels_final": removed_train_only_labels_final,
    "removed_test_only_labels_final": removed_test_only_labels_final,

    "num_train_final": int(len(train_idx)),
    "num_test_final": int(len(test_idx)),

    "train_label_counts_final": dict(Counter(y_all[train_idx])),
    "test_label_counts_final": dict(Counter(y_all[test_idx])),

    "train_evidence_type_counts_final": dict(
        Counter(
            metadata[int(idx)].get("evidence_type", "unknown")
            for idx in train_idx
        )
    ),

    "test_evidence_type_counts_final": dict(
        Counter(
            metadata[int(idx)].get("evidence_type", "unknown")
            for idx in test_idx
        )
    ),
})

print("Final train/test indices prepared.")
print(json.dumps(scene_split_info, indent=2, ensure_ascii=False))

print("\nFinal label order:")
print(label_order)

Final train/test indices prepared.
{
  "train_scenes": [
    "FloorPlan1",
    "FloorPlan2",
    "FloorPlan3",
    "FloorPlan4"
  ],
  "test_scenes": [
    "FloorPlan5",
    "FloorPlan6"
  ],
  "num_train_before_common_label_filter": 2434,
  "num_test_before_common_label_filter": 1348,
  "train_labels_before": [
    "above",
    "below",
    "contains",
    "in",
    "left_of",
    "near",
    "on",
    "right_of",
    "supports"
  ],
  "test_labels_before": [
    "above",
    "below",
    "contains",
    "in",
    "left_of",
    "near",
    "on",
    "right_of",
    "supports"
  ],
  "common_labels_initial": [
    "above",
    "below",
    "contains",
    "in",
    "left_of",
    "near",
    "on",
    "right_of",
    "supports"
  ],
  "removed_train_only_labels_initial": [],
  "removed_test_only_labels_initial": [],
  "num_train_after_initial_common_label_filter": 2434,
  "num_test_after_initial_common_label_filter": 1348,
  "setting": "B3_same_direction_scene_split",
  "direction_fil

In [8]:
# ============================================================
# Build Setting B3 same-direction test subset
#
# In Setting B3 same-direction evaluation, test_idx has already been
# filtered to the same direction as train_idx: direct/direct or inverse/inverse.
# ============================================================

test_subset_key = f"test_{TEST_DIRECTION_SELECTION_MODE}_only"

test_subset_indices = {
    test_subset_key: test_idx,
}

test_subset_summary = {}

for subset_key, idxs in test_subset_indices.items():
    test_subset_summary[subset_key] = {
        "num_examples": int(len(idxs)),
        "label_counts": dict(Counter(y_all[idxs])) if len(idxs) else {},
        "evidence_type_counts": dict(
            Counter(
                metadata[int(idx)].get("evidence_type", "unknown")
                for idx in idxs
            )
        ) if len(idxs) else {},
    }

print("Setting B3 same-direction test subset summary:")
print(json.dumps(test_subset_summary, indent=2, ensure_ascii=False))

if len(test_idx) == 0:
    raise ValueError(f"Setting B3 same-direction test subset is empty: {test_subset_key}")

Setting B3 same-direction test subset summary:
{
  "test_direct_only": {
    "num_examples": 678,
    "label_counts": {
      "above": 73,
      "below": 59,
      "left_of": 161,
      "contains": 5,
      "right_of": 168,
      "near": 54,
      "on": 96,
      "supports": 59,
      "in": 3
    },
    "evidence_type_counts": {
      "explicit_direct": 678
    }
  }
}


In [9]:
# ============================================================
# Layer-wise training and evaluation
# ============================================================

def evaluate_classifier(
    clf: LogisticRegression,
    X_eval: np.ndarray,
    y_eval: np.ndarray,
    labels: List[str],
) -> Dict[str, Any]:
    if len(y_eval) == 0:
        return {
            "num_examples": 0,
            "accuracy": None,
            "macro_f1": None,
            "label_order": labels,
            "classification_report": {},
            "confusion_matrix": [],
        }

    y_pred = clf.predict(X_eval)

    acc = accuracy_score(y_eval, y_pred)
    macro_f1 = f1_score(y_eval, y_pred, labels=labels, average="macro", zero_division=0)

    report = classification_report(
        y_eval,
        y_pred,
        labels=labels,
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(
        y_eval,
        y_pred,
        labels=labels,
    )

    return {
        "num_examples": int(len(y_eval)),
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "label_order": labels,
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
    }


results_by_layer = []

for layer_id in range(num_layers):
    if PRINT_LAYER_PROGRESS:
        print(f"Training layer {layer_id}/{num_layers - 1}")

    X_train_layer = X_all[train_idx, layer_id, :]
    y_train_layer = y_all[train_idx]

    clf = LogisticRegression(
        max_iter=LOGREG_MAX_ITER,
        C=LOGREG_C,
        multi_class="auto",
        solver="lbfgs",
        random_state=RANDOM_SEED,
    )

    clf.fit(X_train_layer, y_train_layer)

    layer_result = {
        "layer": int(layer_id),
        "train": evaluate_classifier(
            clf,
            X_train_layer,
            y_train_layer,
            labels=label_order,
        ),
    }

    for subset_key, idxs in test_subset_indices.items():
        X_eval = X_all[idxs, layer_id, :]
        y_eval = y_all[idxs]

        layer_result[subset_key] = evaluate_classifier(
            clf,
            X_eval,
            y_eval,
            labels=label_order,
        )

    results_by_layer.append(layer_result)

print("Layer-wise training and evaluation complete.")

Training layer 0/36


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Training layer 1/36


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Training layer 2/36


...

Training layer 35/36


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Training layer 36/36


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Layer-wise training and evaluation complete.


In [10]:
# ============================================================
# Summarize top layers
# ============================================================

summary_rows = []

for r in results_by_layer:
    row = {
        "layer": r["layer"],
        "train_accuracy": r["train"]["accuracy"],
        "train_macro_f1": r["train"]["macro_f1"],
    }

    for subset_key in test_subset_indices.keys():
        row[f"{subset_key}_accuracy"] = r[subset_key]["accuracy"]
        row[f"{subset_key}_macro_f1"] = r[subset_key]["macro_f1"]
        row[f"{subset_key}_num_examples"] = r[subset_key]["num_examples"]

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

summary_csv_path = os.path.join(
    STEP6_OUTPUT_DIR,
    f"step6_{EXPERIMENT_NAME}_{MODEL_TAG}_{FEATURE_KEY}_layer_summary.csv",
)

summary_df.to_csv(summary_csv_path, index=False)

print("Saved layer summary CSV:")
print(summary_csv_path)

display(summary_df.head())

if PRINT_TOP_LAYERS:
    metrics_to_print = []

    for subset_key in test_subset_indices.keys():
        metrics_to_print.extend([
            f"{subset_key}_accuracy",
            f"{subset_key}_macro_f1",
        ])

    for metric in metrics_to_print:
        print("\n" + "=" * 80)
        print("Top layers by:", metric)
        print("=" * 80)

        display(
            summary_df.sort_values(
                metric,
                ascending=False,
                na_position="last",
            ).head(NUM_TOP_LAYERS_TO_PRINT)
        )

Saved layer summary CSV:
/content/pilot_4/data/step6_settingB3_scene_split_train_direct_test_direct_outputs/step6_settingB3_scene_split_train_direct_test_direct_Qwen2_5_3B_layer_diff_features_layer_summary.csv


,layer,train_accuracy,train_macro_f1,test_direct_only_accuracy,test_direct_only_macro_f1,test_direct_only_num_examples
0,0,0.655032,0.43939,0.442478,0.303085,678
1,1,1.000000,1.00000,0.792035,0.713818,678
2,2,1.000000,1.00000,0.890855,0.862485,678
3,3,1.000000,1.00000,0.890855,0.860710,678
4,4,1.000000,1.00000,0.877581,0.851395,678



Top layers by: test_direct_only_accuracy


,layer,train_accuracy,train_macro_f1,test_direct_only_accuracy,test_direct_only_macro_f1,test_direct_only_num_examples
5,5,1.0,1.0,0.904130,0.850823,678
17,17,1.0,1.0,0.895280,0.870970,678
19,19,1.0,1.0,0.890855,0.864740,678
3,3,1.0,1.0,0.890855,0.860710,678
2,2,1.0,1.0,0.890855,0.862485,678
4,4,1.0,1.0,0.877581,0.851395,678
6,6,1.0,1.0,0.870206,0.830470,678
18,18,1.0,1.0,0.862832,0.851434,678
20,20,1.0,1.0,0.861357,0.784440,678
21,21,1.0,1.0,0.831858,0.739229,678



Top layers by: test_direct_only_macro_f1


,layer,train_accuracy,train_macro_f1,test_direct_only_accuracy,test_direct_only_macro_f1,test_direct_only_num_examples
17,17,1.0,1.0,0.895280,0.870970,678
19,19,1.0,1.0,0.890855,0.864740,678
2,2,1.0,1.0,0.890855,0.862485,678
3,3,1.0,1.0,0.890855,0.860710,678
18,18,1.0,1.0,0.862832,0.851434,678
4,4,1.0,1.0,0.877581,0.851395,678
5,5,1.0,1.0,0.904130,0.850823,678
6,6,1.0,1.0,0.870206,0.830470,678
7,7,1.0,1.0,0.830383,0.813410,678
10,10,1.0,1.0,0.796460,0.804423,678


In [11]:
# ============================================================
# Save full Step 6 result JSON
# ============================================================

description = (
    "Setting B3 same-direction scene-split evaluation. "
    f"Training uses only {TRAIN_DIRECTION_SELECTION_MODE} samples from train scenes. "
    f"Testing uses only {TEST_DIRECTION_SELECTION_MODE} samples from test scenes. "
    "Scene split is preserved."
)

output_payload = {
    "experiment_name": EXPERIMENT_NAME,
    "description": description,

    "model_name": MODEL_NAME,
    "model_tag": MODEL_TAG,
    "feature_key": FEATURE_KEY,
    "label_field": LABEL_FIELD,

    "num_samples_loaded": int(num_samples),
    "num_layers": int(num_layers),
    "feature_dim": int(feature_dim),

    "label_order": label_order,
    "scene_split_info": scene_split_info,
    "test_subset_summary": test_subset_summary,

    "step6_config": {
        "setting": "B3_same_direction_scene_split",
        "experiment_name": EXPERIMENT_NAME,
        "direction_filter_protocol": DIRECTION_FILTER_PROTOCOL,

        "model_name": MODEL_NAME,
        "model_tag": MODEL_TAG,
        "feature_key": FEATURE_KEY,
        "label_field": LABEL_FIELD,

        "train_scenes": TRAIN_SCENES,
        "test_scenes": TEST_SCENES,

        "filter_train_by_direction": FILTER_TRAIN_BY_DIRECTION,
        "filter_test_by_direction": FILTER_TEST_BY_DIRECTION,

        "train_direction_selection_mode": TRAIN_DIRECTION_SELECTION_MODE,
        "test_direction_selection_mode": TEST_DIRECTION_SELECTION_MODE,

        "logreg_max_iter": LOGREG_MAX_ITER,
        "logreg_c": LOGREG_C,
        "random_seed": RANDOM_SEED,
    },

    "source_pt_files": pt_files,
    "results_by_layer": results_by_layer,
}

output_json_path = os.path.join(
    STEP6_OUTPUT_DIR,
    f"step6_{EXPERIMENT_NAME}_{MODEL_TAG}_{FEATURE_KEY}.json",
)

with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(output_payload, f, indent=2, ensure_ascii=False)

print("Saved full Step 6 result JSON:")
print(output_json_path)

Saved full Step 6 result JSON:
/content/pilot_4/data/step6_settingB3_scene_split_train_direct_test_direct_outputs/step6_settingB3_scene_split_train_direct_test_direct_Qwen2_5_3B_layer_diff_features.json


In [12]:
# ============================================================
# Zip Step 6 outputs
# ============================================================

zip_base = f"/content/pilot4_step6_{EXPERIMENT_NAME}_{MODEL_TAG}_{FEATURE_KEY}"
zip_path = shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir=STEP6_OUTPUT_DIR,
)

print("Created Step 6 output zip:")
print(zip_path)

Created Step 6 output zip:
/content/pilot4_step6_settingB3_scene_split_train_direct_test_direct_Qwen2_5_3B_layer_diff_features.zip


In [13]:
# ============================================================
# Optional download
# ============================================================

from google.colab import files

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>